In [1]:
with open("input.txt", "r", encoding="utf-8") as f:  # we have a dataset of Shakespeare
    text = f.read()

print(f"length of dataset in characters: {len(text)}")

length of dataset in characters: 1115394


In [2]:
print(text[:1000])  # looking at first 1000 characters

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [3]:
# we are printing all the special/unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
# create a mapping from strings(words) to numbers and vice-versa
stoi = { ch:i for i,ch in enumerate(chars) }  # stoi = string to integer
itos = { i:ch for i,ch in enumerate(chars) }  # itos = integer to string
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hii foks"))
print(decode(encode("hii foks")))

[46, 47, 47, 1, 44, 53, 49, 57]
hii foks


In [5]:
# encoding the entire text into a single tensor(array)
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])  # the 1000 characters we look at earlier, the GPT will look to it like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [6]:
# let's now split up the data into train and validation sets
n = int(0.9 * len(data))  # we will train the first 90% of the data, and the remaining data of the last to with-hold
train_data = data[:n]  # from 0 to n
val_data = data[n:]   # validation data (from n to last)

In [7]:
# IMP
block_size = 8  # we will feed the data in chunks not all at once all. which will help it to leaning and adapt the text
chunk = train_data[:block_size+1]   # block_size+1 helps, as the model will predict the next character
                                    # by taking the help of the previous all characters
print(chunk)

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [8]:
x = chunk[:block_size]
y = chunk[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]     # this was the same thing, the model will take help of the previous all the character to predict the
    target = y[t]         # next one
    print(f"when the input is {context} the target(output) is:{target}")


when the input is tensor([18]) the target(output) is:47
when the input is tensor([18, 47]) the target(output) is:56
when the input is tensor([18, 47, 56]) the target(output) is:57
when the input is tensor([18, 47, 56, 57]) the target(output) is:58
when the input is tensor([18, 47, 56, 57, 58]) the target(output) is:1
when the input is tensor([18, 47, 56, 57, 58,  1]) the target(output) is:15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) the target(output) is:47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target(output) is:58


In [9]:
torch.manual_seed(1337)
batch_size = 4  # how many independent sequence we will take for training/processing in parallel?
block_size = 8  # what will be the maximum context length of that predictions

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # ix = position/index, it will pick random position from the whole training data chunks, whose range is from low=0 to high=data length - block size, and it will give a tensor of size 4, of positions which can be taken from any small part of the text

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# V IMP
xb, yb = get_batch("train")   # so, 4*8 = 32 different character which will be feed as input to the transformer
                              # position 1 prediction, position 2 prediction, and so on --> in total 32
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)
print()
for b in range(batch_size):    # batch dimension   it's basically, making a matrix
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]     # b = row, t = column
        print(f"for the input{context.tolist()} target is: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])

for the input[24] target is: 43
for the input[24, 43] target is: 58
for the input[24, 43, 58] target is: 5
for the input[24, 43, 58, 5] target is: 57
for the input[24, 43, 58, 5, 57] target is: 1
for the input[24, 43, 58, 5, 57, 1] target is: 46
for the input[24, 43, 58, 5, 57, 1, 46] target is: 43
for the input[24, 43, 58, 5, 57, 1, 46, 43] target is: 39
for the input[44] target is: 53
for the input[44, 53] target is: 56
for the input[44, 53, 56] target is: 1
for the input[44, 53, 56, 1] target is: 58
for the input[44, 53, 56, 1, 58] target is: 46
for the input[44, 53, 56, 1, 58, 46] target is: 39


In [10]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        # step 1: initialize the parent nn.Module internal tracking
        super().__init__()

        # step 2: defining the layers with --> V IMP trainable weights
        # each token directly reads off the logits(predictions) for the next token from a look-up table which is: IMP the Embedding Table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # 65 cross 65 table

    def forward(self, idx, targets=None):
        # step 3: Forward pass --> how data will flow in the layers of the model


        logits = self.token_embedding_table(idx)   # logist = raw outputs from lst layers from the neural network before sigmoid function, i.e. before hitting the softmax functions
        # logits will have shape(size) of = (B,T,C)

        # but the pytorch needs output for loss calculation needs in (N,C) dim
        # N = total predictions

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape   # un-packing them : B = 4, T = 8, C = 65
            # B = batch size, T = time/sequence length (block size), C = channels (vocab size)

            # Re-Shape logits(B,T,C) to --> (B*T,C) = (4*8, 65) i.e. (32, 65) --> 32 is the total inputs in a 1 particular batch
            logits = logits.view(B*T, C)

            # Re-Shape targets(B,T) to --> (B*T)
            # V IMP : first the targets were a (4, 8) tensor, but we need a 1D single tensor i.e. (B*T) = (4*8) = (32)
            targets = targets.view(B*T)

            # V IMP:  loss calculation
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_token):
        # idx is a (B,T) tensor of token ID's in the current context
        # idx is the input given which is in the form of (B, T) tensor of token ID's
        for _ in range(max_new_token):
            # step 1: Forward pass to get prediction
            logits, loss = self(idx)  # logits will be of size (B, T, C) and target = None

            # step 2: Plucking out the very last character's logits
            logits = logits[:, -1, :]  # Shape changes from (B, T, C) to (B, C)

            # step 3: Converting the logits into probabilities using softmax
            probs = F.softmax(logits, dim=-1)  # Shape:(B, C)
            # dim=-1 tells do probability along the row and not the cols, so at end all the elements along the row probability sums to 1

            # STEP 4: Sample 1 next token from the probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # Shape: (B, 1)
            # the reason why shape is (B, 1) cause we have taken a single number form each batch row
            # and each batch row represent separate sentences

            # step 5: append the newly generated token in the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # Shape: (B, T+1)

        return idx


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [12]:
# Looking how the step works

#print(logits)
#print(F.softmax(logits, dim=-1))
#print(torch.multinomial(F.softmax(logits, dim=-1), num_samples=1))

In [13]:
# Creating a Model
m = BigramLanguageModel(vocab_size)

# Start with a single 0 token (newline character)
#idx = torch.zeros((1, 1), dtype=torch.long)

# Generate 100 new characters
generate_tokens = m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_token=100)

# Decode integers back to text using itos
print(decode(generate_tokens[0].tolist()))


v?vqKGh
txYXcDRSW&jxn;P Kuvbv
Bl:KuMWi;JYlFahlK3u.M,QV;oky3wcxoYPQpbpb
r3Nc 'MQHQ uOfxoxwXIvSE-$&a$K


In [14]:
# 1. Create the optimizer (AdamW is the standard modern optimizer)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32

# Train for 10,000 steps
for steps in range(10000):

    # sample a batch of data
    xb, yb = get_batch("train")

    # Forward pass: evaluate the loss
    logits, loss = m(xb, yb)

    # --- The 3 Golden Steps of PyTorch Training ---
    optimizer.zero_grad(set_to_none=True)  # Re-set previous gradients
    loss.backward()                        # Calculate new gradients (backprop)
    optimizer.step()                       # Update model parameters

# this is the final loss generated
print(loss.item())

2.3781542778015137


In [15]:
#print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_token=100)[0].tolist()))

generate_tokens = m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_token=500)
print(decode(generate_tokens[0].tolist()))


Yorowathit comoue int at se,
Crrtr mou oneny grom be;
mva-BUGR in y llican fer wid.


MIt
O:


OLAfobont aiee;
Anlone aves ve t fo mnd h ve g ay' th,
He heng-piur Yom: nee
MPrny
haiolersceangen
Bur we harplo ho pleich thalk wiriteamithe in, ow
Jute mjeneno hard SHO:

Tht bith;

An: se'
Ox, sodout ws myot:
Fothar, whe shef; pr,
Hesith
pu wotif$w brurot pinou banks y whocois arioofunce trd thir omiththio tas ofe, aly d beanerok GSclaves.
Y shay wacometend hy, s.

Whel ove bu grdde.
wolke IO: ane l


In [16]:
# The Mathematical Trick for Self-Attention

In [17]:
# Consider the following toy example
torch.manual_seed(1337)

B, T, C = 4, 8, 2 # batch, time, channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [18]:
# Version 1: using simple loop, to calculate the avg of the previous tokens
# We want mean of x[b, t} = mean_{i<=t} x[b, i]
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b, t] = torch.mean(xprev, 0)

In [19]:
# Version 2: using Matrix, to calculate the avg of the previous tokens
wei = torch.tril(torch.ones(T, T))  # The reason why we took (T, T) is we are having total 8 token to calculate it's average
wei = wei / wei.sum(1, keepdim=True)

# V IMP
# So we are multiplying (T, T) to (B, T, C) i.e. (8, 8) @ (4, 8, 2) but, B is the total different batch, pytorch will not take it so we have to pass same batch dimension also with (T, T)
# So it will be (B, T, T) @ (B, T, C) = which will give (B, T, C) matrix of the average info, and shape will be same of (B, T, C)

xbow2 = wei @ x # (B, T, T) @ (B, T, C) = (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6)   # atol (Looser tolerance) = 0.000001

True

In [20]:
# Version 3: using softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)   # Softmax does the same thing by dividing by wei
xbow3 = wei @ x
torch.allclose(xbow, xbow3, atol=1e-6)

True

In [21]:
# Version 4: Self-attention
torch.manual_seed(1337)

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# Let's see a single head perform the Self Attention
head_size = 16

# V IMP : Linear helps to create two separate Query and Key layer in neural network
# 1. Project into Q, K, V
key = nn.Linear(C, head_size, bias=False)   # 32--> 16   # Either pass C (channels) or the n_embd as both now are 32 in size
query = nn.Linear(C, head_size, bias=False) # 32--> 16
value = nn.Linear(C, head_size, bias=False) # 32--> 16    <-- this is input similar to 'b'

# V IMP

k = key(x)   # (B, T=8, C=16)  as we have divided into two parts, hence, H = size of the single head = 16
q = query(x)   # (B, T=8, C=16)

# 2. Compute attention scores ("affinities")
wei = q @ k.transpose(-2, -1)   # matrix-multiply   (B=4, T=8, H=C=16) @ (B=4, H=C=16, T=8) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))

# 3. Prevent looking into the future
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

# 4. Weighted aggregation of values
v = value(x)
out = wei @ v    # (B, T, T) @ (B, T, 16) --> (B, T, 16)
#out = wei @ x

In [22]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089

In [23]:
out

tensor([[[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
           7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9068e-01,
           1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8598e-01,  2.8623e-01,
           5.7099e-01],
         [ 6.7643e-01, -5.4770e-01, -2.4780e-01,  3.1430e-01, -1.2799e-01,
          -2.9521e-01, -4.2962e-01, -1.0891e-01, -4.9282e-02,  7.2679e-01,
           7.1296e-01, -1.1639e-01,  3.2665e-01,  3.4315e-01, -7.0975e-02,
           1.2716e+00],
         [ 4.8227e-01, -1.0688e-01, -4.0555e-01,  1.7696e-01,  1.5811e-01,
          -1.6967e-01,  1.6217e-02,  2.1509e-02, -2.4903e-01, -3.7725e-01,
           2.7867e-01,  1.6295e-01, -2.8951e-01, -6.7610e-02, -1.4162e-01,
           1.2194e+00],
         [ 1.9708e-01,  2.8561e-01, -1.3028e-01, -2.6552e-01,  6.6781e-02,
           1.9535e-01,  2.8073e-02, -2.4511e-01, -4.6466e-01,  6.9287e-02,
           1.5284e-01, -2.0324e-01, -2.4789e-01, -1.6213e-01,  1.9474e-01,
           7.6778e-01],
    

In [24]:
wei   # wei = by version 3

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089

In [25]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b
print("a=")
print(a)
print("b=")
print(b)
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [28]:
class BatchNorm1d:

    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)  # learnable scale
        self.beta = torch.zeros(dim)  # learnable shift

    def __call__(self, x):
        # Calculate mean and variance across BATCH (dimension = 0)
        xmean = x.mean(1, keepdim=True)  # Batch mean
        xvar = x.var(1, keepdim=True)    # Batch Variance

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


In [29]:
torch.manual_seed(1337)

module = BatchNorm1d(100)
x = torch.randn(32, 100) # batch of 32 examples, each with 100 dimensions
x = module(x)

print(x.shape)
print(x[:, 0].mean(), x[:, 0].var())   # output : 0.0, 1.0   normalized down 0th column
print(x[0, :].mean(), x[0, :].var())

torch.Size([32, 100])
tensor(0.1469) tensor(0.7750)
tensor(-9.5367e-09) tensor(1.0000)
